In [2]:
import pandas as pd
from typing import List, Dict, Optional, Any, TypedDict
import re
from pydantic import BaseModel, Field, ConfigDict


class DatasetMetadata(BaseModel):
    name: str
    path: str
    profile_summary: Optional[Dict[str, Any]] = None
    schema: Optional[List[str]] = None

class SchemaMatch(BaseModel):
    dataset_a: str
    dataset_b: str
    matches: Optional[List[Dict[str, Any]]] = None
    generated_code: Optional[str] = None

class Issue(BaseModel):
    description: str
    severity: str = "medium"
    agent: Optional[str] = None
    resolved: bool = False

class IntegrationState(BaseModel):
    """
    Central state shared across all agents.
    Tracks datasets, schema matches, issues, and final fused output.
    """
    # FIX: Allow Pydantic to handle the complex pd.DataFrame type
    model_config = ConfigDict(arbitrary_types_allowed=True)

    datasets: List[DatasetMetadata] = Field(default_factory=list)
    schema_matches: List[SchemaMatch] = Field(default_factory=list)
    issues: List[Issue] = Field(default_factory=list)
    fused_data: Optional[pd.DataFrame] = None

class AgentState(TypedDict):
    """The state of the graph, passed between nodes."""
    input_message: str
    schema_code: str
    resolution_code: str
    validator_output: str
    max_retries: int
    current_retry: int
    validation_status: str

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DatasetMetadata" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [3]:
from PyDI.profiling import DataProfiler
from PyDI.io import load_csv, load_xml, load_parquet
# from langchain_groq import ChatGroq
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

# --- UTILITY FUNCTION ---
def extract_python_code(response_text: str) -> Optional[str]:
    """Extracts the Python code block from the LLM response."""
    # Regex to find content inside triple backticks with optional 'python' marker
    match = re.search(r"```python\s*(.*?)\s*```", response_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# --- AGENT TOOLS ---

# TOOL 1: Dataset Loader + Profiler
@tool
def load_and_profile_dataset(file_path: str) -> str:
    """
    Loads dataset (CSV, Parquet, XML) and profiles it using PyDI DataProfiler.
    The output is a JSON string summarizing the dataset's profile and a 3-row sample.
    """
    ext = os.path.splitext(file_path)[1].lower()
    df = None
    try:
        if ext == ".csv":
            df = load_csv(file_path)
        elif ext == ".parquet":
            df = load_parquet(file_path)
        elif ext == ".xml":
            df = load_xml(file_path)
        else:
            raise ValueError(f"Unsupported format: {ext}. Supported: CSV, Parquet, XML.")

        if df is None or df.empty:
             return f"Error: Dataset at {file_path} loaded as empty or failed to load."

        profiler = DataProfiler()
        profile = profiler.summary(df)

        return json.dumps({
            "file_path": file_path,
            "profile": profile,
            "sample": df.head(3).to_dict(orient="records")
        }, indent=2)
    except Exception as e:
        return f"Error loading or profiling dataset at {file_path}: {e}"

# TOOL 2: PyDI Docs Retriever
@tool
def get_pydi_doc(module_name: str) -> str:
    """
    Fetch documentation for a PyDI module or class.
    Example usage: get_pydi_doc('PyDI.schemamatching.SchemaMatcher')
    """
    try:
        components = module_name.split(".")
        module = __import__(components[0])
        for comp in components[1:]:
            module = getattr(module, comp)
        return module.__doc__ or f"No docstring available for {module_name}."
    except Exception as e:
        return f"Error retrieving docs for {module_name}: {e}"

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END


llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=512
)

integration_tools = [load_and_profile_dataset, get_pydi_doc]

# 1. SCHEMA INTEGRATION AGENT (Code Generator)

# integration_agent_prompt = hub.pull("hwchase17/structured-chat-agent")

# Define a custom prompt suitable for create_tool_calling_agent
integration_agent_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant with access to tools."),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

integration_agent_runnable = create_tool_calling_agent(llm, integration_tools, integration_agent_prompt)

integration_agent_executor = AgentExecutor(
    agent=integration_agent_runnable,
    tools=integration_tools,
    verbose=True,
    handle_parsing_errors=True
)

# 2. CODE VALIDATOR AGENT (Code Debugger)
# This agent does not need tools; its job is pure reasoning/critique.
validator_prompt_template = """
You are an expert Python Code Validator and Debugger.

Your current task is to review and fix the PyDI Python script below.
Assume the code was run and failed with a generic error (e.g., missing import, incorrect function call, syntax mistake).

The code generated is:
--- CODE START ---
{generated_code}
--- CODE END ---

**Self-Healing/Fixing Instructions:**
1.  **Critique:** Review the code for all common errors (syntax, logic, missing imports, PyDI method signatures).
2.  **Fix:** If an error is found, rewrite the entire, corrected, fully working Python script.
3.  **Validation:** If the code appears perfect and ready to run, respond **ONLY** with the phrase "CODE_OK".

Respond ONLY with the complete, corrected Python code inside triple backticks (```python ... ```) or the exact phrase "CODE_OK". Do not include any other text or reasoning.
"""
validator_prompt = ChatPromptTemplate.from_template(validator_prompt_template)
validator_chain = validator_prompt | llm | StrOutputParser()

ImportError: cannot import name 'ContextT' from 'langgraph.typing' (/usr/local/lib/python3.12/dist-packages/langgraph/typing.py)

In [ ]:
# 3. IDENTITY RESOLUTION AGENT (Blocking & Matching Code Generator)
resolution_prompt_template = """
You are an expert PyDI Entity Resolution Agent specializing in blocking and matching.

A previous agent successfully generated and validated the **Schema Matching** code.
Your goal is to complete the pipeline by generating the code for **Identity Resolution**.

**Context (Validated Schema Matching Code):**
--- PREVIOUS CODE START ---
{validated_schema_match_code}
--- PREVIOUS CODE END ---

**Your Task:**
1.  **Assume** the code in the 'PREVIOUS CODE' block runs successfully and variables like `df_a`, `df_b`, and the schema mapping result are available.
2.  **Use your PyDI knowledge and the 'get_pydi_doc' tool** to determine the optimal blocking and matching methods (e.g., `StandardBlocker`, `SortedNeighbourhoodBlocker`, `TokenBlocker`) based on the assumed schema.
3.  **Generate a complete, fully executable PyDI Python script** that performs the *entire* workflow:
    * Imports (from PyDI.io, PyDI.profiling, PyDI.entitymatching, etc.)
    * Dataset Loading (using the provided paths)
    * **Schema Matching (Integrate the validated code)**
    * **Blocking (Blocking setup and execution)**
    * **Matching/Pairing (Matcher setup and execution)**

Respond ONLY with the complete, end-to-end Python script inside triple backticks.
"""
resolution_agent_runnable = create_tool_calling_agent(llm, integration_tools, integration_agent_prompt.partial(
    system=resolution_prompt_template
))
resolution_agent_executor = AgentExecutor(
    agent=resolution_agent_runnable,
    tools=integration_tools,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
def generate_schema_code(state: AgentState) -> dict:
    """Node 1: Generates initial Schema Matching code."""
    print(f"\n[NODE] Running Schema Generator (Attempt {state.get('current_retry', 1)})")

    integration_task_description = f"""
    You are an expert Autonomous Schema Integration Agent.
    Your goal is to write minimal working PyDI code for **initial dataset loading and schema matching**.
    Datasets: {state['input_message']}.
    Begin by calling your tool to profile the first dataset mentioned.
    """

    response = integration_agent_executor.invoke({"input": integration_task_description})
    initial_code = extract_python_code(response.get("output", ""))

    return {"schema_code": initial_code or "Failed to generate code."}


def validate_schema_code(state: AgentState) -> dict:
    """Node 2: Validates the generated Schema Matching code."""
    print("\n[NODE] Running Validator on Schema Code...")
    code_to_validate = state["schema_code"]

    if not code_to_validate or "Failed to generate code." in code_to_validate:
        return {"validation_status": "FAILED", "validator_output": "No code generated."}

    validation_input = {"generated_code": code_to_validate}
    validation_result = validator_chain.invoke(validation_input)

    if validation_result.strip() == "CODE_OK":
        print("[VALIDATOR] Schema Code Confirmed: CODE_OK")
        return {"validation_status": "CODE_OK", "validator_output": "CODE_OK", "current_retry": 0}

    fixed_code = extract_python_code(validation_result)

    if fixed_code:
        print("[VALIDATOR] Schema Code Fixed: Retrying generation step with fixed code as context.")
        return {
            "validation_status": "CODE_FIXED",
            "schema_code": fixed_code,
            "current_retry": state.get('current_retry', 1) + 1
        }

    print("[VALIDATOR] Schema Code Validation Failed: No clean fix provided.")
    return {"validation_status": "FAILED"}


def generate_resolution_code(state: AgentState) -> dict:
    """Node 3: Generates Identity Resolution code using validated Schema code."""
    print("\n[NODE] Running Resolution Generator.")

    resolution_prompt_context = resolution_prompt_template.format(
        validated_schema_match_code=state["schema_code"]
    )

    response = resolution_agent_executor.invoke({"input": resolution_prompt_context})
    resolution_code = extract_python_code(response.get("output", ""))

    return {"resolution_code": resolution_code or "Failed to generate resolution code."}


def validate_resolution_code(state: AgentState) -> dict:
    """Node 4: Validates the generated Identity Resolution code."""
    print("\n[NODE] Running Validator on Resolution Code...")
    code_to_validate = state["resolution_code"]

    if not code_to_validate or "Failed to generate resolution code." in code_to_validate:
        return {"validation_status": "FAILED", "validator_output": "No resolution code generated."}

    validation_input = {"generated_code": code_to_validate}
    validation_result = validator_chain.invoke(validation_input)

    if validation_result.strip() == "CODE_OK":
        print("[VALIDATOR] Resolution Code Confirmed: CODE_OK")
        return {"validation_status": "CODE_OK", "validator_output": "CODE_OK"}

    fixed_code = extract_python_code(validation_result)

    # We use a special key here to track retries for the resolution phase separately
    resolution_retries = state.get('resolution_retries', 0) + 1

    if fixed_code and resolution_retries <= state['max_retries']:
        print(f"[VALIDATOR] Resolution Code Fixed (Attempt {resolution_retries}/{state['max_retries']}): Retrying resolution generation.")
        return {
            "validation_status": "CODE_FIXED",
            "resolution_code": fixed_code,
            "resolution_retries": resolution_retries
        }

    print("[VALIDATOR] Resolution Code Validation Failed: Max retries reached or no clean fix.")
    return {"validation_status": "FAILED"}


# --- LANGGRAPH CONDITIONAL EDGES ---

def decide_schema_fix(state: AgentState) -> str:
    """Decide whether to fix/retry Schema generation or move to Resolution."""
    status = state["validation_status"]
    current_retry = state.get('current_retry', 1)

    if status == "CODE_OK":
        print("[DECISION] Schema Code Validated. Moving to Resolution Generator.")
        return "generate_resolution_code"

    if status == "CODE_FIXED" and current_retry < state["max_retries"]:
        print(f"[DECISION] Schema Code Fixed. Retrying Schema Generator (Attempt {current_retry}).")
        return "generate_schema_code"

    print("[DECISION] Schema Code Failed (Max retries/Failure). Ending pipeline.")
    return END


def decide_resolution_fix(state: AgentState) -> str:
    """Decide whether to fix/retry Resolution generation or finish."""
    status = state["validation_status"]
    resolution_retries = state.get('resolution_retries', 1)

    if status == "CODE_OK":
        print("[DECISION] Resolution Code Validated. Ending pipeline.")
        return END

    if status == "CODE_FIXED" and resolution_retries < state["max_retries"]:
        print(f"[DECISION] Resolution Code Fixed. Retrying Resolution Generator (Attempt {resolution_retries}).")
        return "generate_resolution_code"

    print("[DECISION] Resolution Code Failed (Max retries/Failure). Ending pipeline.")
    return END

In [ ]:
graph = StateGraph(AgentState)

# Add Nodes
graph.add_node("generate_schema_code", generate_schema_code)
graph.add_node("validate_schema_code", validate_schema_code)
graph.add_node("generate_resolution_code", generate_resolution_code)
graph.add_node("validate_resolution_code", validate_resolution_code)

# Build the Schema Flow (with self-healing loop)
graph.set_entry_point("generate_schema_code")
graph.add_edge("generate_schema_code", "validate_schema_code")
graph.add_conditional_edges("validate_schema_code", decide_schema_fix)

# Build the Resolution Flow (with self-healing loop)
graph.add_edge("generate_resolution_code", "validate_resolution_code")
graph.add_conditional_edges("validate_resolution_code", decide_resolution_fix)

# Compile the Graph
app = graph.compile()


# --- EXECUTION ---

TASK_INPUT = {
    "dataset_a_path": "datasets/amazon.patquet",
    "dataset_b_path": "datasets/goodreads.patquet",
}

initial_state: AgentState = {
    "input_message": f"Datasets: {TASK_INPUT['dataset_a_path']}, {TASK_INPUT['dataset_b_path']}, and {TASK_INPUT['dataset_c_path']}",
    "schema_code": "",
    "resolution_code": "",
    "validator_output": "",
    "max_retries": 3, # Maximum number of times to try fixing code
    "current_retry": 1,
    "validation_status": "",
    "resolution_retries": 1,
}

print("--- STARTING LANGGRAPH EXECUTION ---")
print("Target: Schema Matching and Identity Resolution PyDI Pipeline.")

# Run the graph
final_state = app.invoke(initial_state)

print(f"Final Schema Code Status: {final_state['validation_status']}")

# Check if the pipeline completed successfully
if final_state["validation_status"] == "CODE_OK" and final_state["resolution_code"]:
    print("\nFINAL VALIDATED PYDI PIPELINE SCRIPT:")
    print("--"*50)
    print(final_state['resolution_code'])
    print("--"*50)
else:
    print("\n[FAILURE] Pipeline failed to produce validated Identity Resolution code after max retries.")
    print("Last generated Schema Code:")
    print(final_state['schema_code'])
    print("\nLast generated Resolution Code (May be incomplete/invalid):")
    print(final_state['resolution_code'])